# Islas de calor urbanas y El Niño: flujo metodológico en ArcGIS Pro

**Alcance y advertencia.** Aquí se replica un escenario de análisis sencillo, con fines didácticos y metodológicos. Cada análisis territorial tiene sus particularidades (zona, sensor, fechas, datos de entrada, normativa local), y los parámetros usados (pesos, cortes, distancias, unidad mínima de mapeo) deben revisarse y justificarse en cada caso. Para un estudio más robusto conviene tener en cuenta, con precaución:

- Una o pocas fechas no representan el comportamiento térmico de un territorio. Lo recomendable es usar varias escenas por época y series de tiempo.
- La temperatura de superficie (LST) no es la temperatura del aire ni equivale a la exposición térmica de las personas.
- Al comparar fechas se requiere el mismo sensor, la misma época del año, nubosidad baja y el mismo procesamiento. Diferencias de varios grados suelen indicar escenas no comparables y no un efecto climático.
- Una sola comparación no demuestra causalidad. La expansión urbana, los cambios de cobertura y el clima del día también cambian la LST.
- Los pesos del proceso analítico jerárquico son subjetivos: conviene validarlos con expertos y hacer un análisis de sensibilidad.
- Los resultados deben contrastarse con datos de campo o estaciones, y revisar el tratamiento de cuerpos de agua, vías y nubes delgadas.
- Los productos de este análisis apoyan la decisión y no reemplazan estudios técnicos detallados.

Requisitos: ArcGIS Pro 3.x con la extensión Spatial Analyst.

Datos de entrada:

- Escenas de reflectancia y temperatura de superficie con corrección atmosférica, una carpeta por escena. Se usan las bandas roja, infrarroja cercana y térmica, y la banda de calidad de píxel.
- Áreas de actividad del POT con el campo de uso principal (`AreasActividad_POT`).
- Límite del suelo urbano y de expansión (`Limite_UrbanoExpansion`), que sirve de máscara del análisis y de fuente del suelo de expansión sin uso definido.

Las dos capas vectoriales deben estar dentro de la geodatabase definida en los parámetros y en MAGNA-SIRGAS Origen Nacional (EPSG:9377).

Flujo:

1. NDVI y temperatura superficial (LST).
2. Reclasificación a cinco niveles y combinación lineal ponderada.
3. Cruce con los usos del suelo del POT y matriz de decisión.
4. Análisis de puntos calientes Getis-Ord Gi*.
5. Unidad mínima de mapeo y núcleos de intervención por nivel de atención.
6. Comparación de escenarios.

In [ ]:
import os
import arcpy
import numpy as np
from arcpy.sa import *

arcpy.CheckOutExtension("Spatial")

gdb = r"D:\RUTA\DEMO_BAQ.gdb"
escenas = [
    r"D:\RUTA\DEMO_Fenomeno_ElNino\LC09_L2SP\escena1",
    r"D:\RUTA\DEMO_Fenomeno_ElNino\LC09_L2SP\escena2"]
fc_areas_actividad = os.path.join(gdb, "AreasActividad_POT")
fc_expansion = os.path.join(gdb, "Limite_UrbanoExpansion")
fc_limite = os.path.join(gdb, "Limite_UrbanoExpansion")
campo_uso = "Uso_Principal"

escala_sr = 0.0000275
desplazamiento_sr = -0.2
escala_st = 0.00341802
desplazamiento_st = 149.0

if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(os.path.dirname(gdb), os.path.basename(gdb))

arcpy.env.workspace = gdb
arcpy.env.scratchWorkspace = gdb
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(9377)
arcpy.env.cellSize = 30
arcpy.env.extent = fc_limite
arcpy.env.mask = fc_limite

## 1. NDVI y temperatura superficial

Los productos de superficie entregan reflectancia (banda roja `SR_B4` e infrarroja cercana `SR_B5`) y temperatura (`ST_B10`) ya corregidas atmosféricamente, incluida la emisividad. Solo hay que aplicar los factores de escala que define el proveedor del producto; los valores usados aquí son los de ese producto y deben verificarse si se cambia de fuente.

Reflectancia de superficie:

$$\rho = 0{,}0000275 \cdot DN - 0{,}2$$

Temperatura de superficie en kelvin y en grados Celsius:

$$ST_K = 0{,}00341802 \cdot DN + 149{,}0 \qquad LST = ST_K - 273{,}15$$

Índice de vegetación de diferencia normalizada:

$$NDVI = \frac{\rho_{NIR} - \rho_{RED}}{\rho_{NIR} + \rho_{RED}}$$

Se descartan los píxeles con relleno, nube, sombra de nube y cirros usando los bits 0 a 4 de `QA_PIXEL`, y los de reflectancia fuera del rango físico válido de 0 a 1, que producirían NDVI menores a -1 o mayores a 1. Si se indican varias escenas, el NDVI y la LST finales son el promedio de los píxeles válidos de cada una.

Los resultados se agregan al mapa con una rampa de color estirada entre el mínimo y el máximo, para ver el rango completo de valores.

In [ ]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
mapa = aprx.listMaps()[0]


arcpy.env.addOutputsToMap = False
creadas = set()


def quitar_capas(nombres):
    for m in aprx.listMaps():
        for capa in m.listLayers():
            if capa.name in nombres:
                m.removeLayer(capa)


def mostrar(nombre, rampa="Condition Number"):
    quitar_capas([nombre])
    capa = mapa.addDataFromPath(os.path.join(gdb, nombre))
    simbologia = capa.symbology
    simbologia.updateColorizer("RasterStretchColorizer")
    simbologia.colorizer.stretchType = "MinimumMaximum"
    simbologia.colorizer.colorRamp = (aprx.listColorRamps(rampa) or aprx.listColorRamps())[0]
    capa.symbology = simbologia
    creadas.add(nombre)


def agregar(nombre):
    quitar_capas([nombre])
    capa = mapa.addDataFromPath(os.path.join(gdb, nombre))
    creadas.add(nombre)
    return capa


def dejar_solo(conservar):
    for capa in mapa.listLayers():
        if capa.name in creadas and capa.name not in conservar:
            mapa.removeLayer(capa)

In [ ]:
def propiedad(raster, clave):
    return float(arcpy.management.GetRasterProperties(raster, clave).getOutput(0).replace(",", "."))


def calcular_escena(carpeta, etiqueta):
    def archivo(sufijo):
        return os.path.join(carpeta, [f for f in os.listdir(carpeta) if f.endswith(sufijo)][0])

    def reflectancia(sufijo):
        valor = Raster(archivo(sufijo)) * escala_sr + desplazamiento_sr
        return SetNull((valor < 0) | (valor > 1), valor)

    invalido = BitwiseAnd(Raster(archivo("_QA_PIXEL.TIF")), 31) > 0
    rojo = reflectancia("_SR_B4.TIF")
    nir = reflectancia("_SR_B5.TIF")
    ndvi = SetNull(invalido, (nir - rojo) / (nir + rojo))
    lst = SetNull(invalido, Raster(archivo("_ST_B10.TIF")) * escala_st + desplazamiento_st - 273.15)
    ndvi.save(etiqueta + "_ndvi")
    lst.save(etiqueta + "_lst")
    return etiqueta + "_ndvi", etiqueta + "_lst"

In [ ]:
quitar_capas(["ndvi_base", "lst_base"])

resultados = [calcular_escena(carpeta, "e%d" % (i + 1)) for i, carpeta in enumerate(escenas)]

CellStatistics([r[0] for r in resultados], "MEAN").save("ndvi_base")
CellStatistics([r[1] for r in resultados], "MEAN").save("lst_base")

for nombre in ("ndvi_base", "lst_base"):
    print("{}: minimo {:.2f}, maximo {:.2f}, media {:.2f}".format(
        nombre, propiedad(nombre, "MINIMUM"), propiedad(nombre, "MAXIMUM"), propiedad(nombre, "MEAN")))

mostrar("lst_base")
mostrar("ndvi_base")

## 2. Reclasificación y combinación lineal ponderada

Cada variable se lleva a una escala común de 1 a 5, donde 1 es prioridad muy baja (menor temperatura y más vegetación) y 5 es prioridad muy alta (mayor temperatura y suelo sin vegetación).

- LST: cinco clases por cortes naturales (Jenks). Clase 1 es la más fría y clase 5 la más caliente.
- NDVI: cinco clases, invertidas. El suelo desnudo recibe 5 y la vegetación densa recibe 1. Los límites se calculan como quintiles de la imagen, porque los rangos fijos de la literatura dependen del producto, el sensor y la escena. Para comparar escenarios hay que reutilizar los mismos valores de `limites_ndvi`.

Los pesos se obtienen con el proceso analítico jerárquico de Saaty (1977): se normaliza cada columna de la matriz de comparación por pares y se promedia por filas.

$$w_i = \frac{1}{n}\sum_{j=1}^{n}\frac{a_{ij}}{\sum_{k=1}^{n}a_{kj}}$$

Con LST una vez y media más importante que NDVI, resultan $w_{LST}=0{,}60$ y $w_{NDVI}=0{,}40$.

Combinación lineal ponderada:

$$S = \sum_{i=1}^{n} w_i x_i \qquad S = 0{,}60 \cdot LST_{norm} + 0{,}40 \cdot NDVI_{norm}$$

El índice continuo `S` va de 1 a 5 y se divide en cinco niveles de igual amplitud: muy bajo, bajo, moderado, alto y muy alto.

In [ ]:
comparacion = np.array([[1.0, 1.5], [1 / 1.5, 1.0]])
pesos = (comparacion / comparacion.sum(axis=0)).mean(axis=1)
peso_lst, peso_ndvi = float(pesos[0]), float(pesos[1])
print("Peso LST {:.2f}, peso NDVI {:.2f}".format(peso_lst, peso_ndvi))

In [ ]:
valores_ndvi = arcpy.RasterToNumPyArray("ndvi_base", nodata_to_value=np.nan)
limites_ndvi = [round(float(v), 2) for v in np.nanpercentile(valores_ndvi, [20, 40, 60, 80])]
print("Límites de NDVI:", limites_ndvi)

quitar_capas(["prioridad_continua"])
dejar_solo(["lst_base", "ndvi_base"])

lst_norm = Slice("lst_base", 5, "NATURAL_BREAKS")
ndvi_norm = Reclassify("ndvi_base", "Value", RemapRange([
    [-1.01, limites_ndvi[0], 5],
    [limites_ndvi[0], limites_ndvi[1], 4],
    [limites_ndvi[1], limites_ndvi[2], 3],
    [limites_ndvi[2], limites_ndvi[3], 2],
    [limites_ndvi[3], 1.01, 1]]), "DATA")

lst_norm.save("lst_norm")
ndvi_norm.save("ndvi_norm")

prioridad = lst_norm * peso_lst + ndvi_norm * peso_ndvi
prioridad.save("prioridad_continua")
mostrar("prioridad_continua")

nivel = Reclassify(prioridad, "Value", RemapRange([
    [0.99, 1.8, 1], [1.8, 2.6, 2], [2.6, 3.4, 3], [3.4, 4.2, 4], [4.2, 5.01, 5]]), "DATA")
nivel.save("prioridad_nivel")

## 3. Cruce con los usos del suelo del POT

La parte del suelo de expansión que no cubren las áreas de actividad se anexa a ellas, sin uso definido, para tener una sola capa de análisis. El ráster de niveles se convierte a polígonos y se cruza con esa capa, de modo que cada fragmento tenga su nivel de prioridad y su uso principal.

La matriz de decisión incorpora la exposición de la población. Residencial, Mixto, Institucional y Dotacional son de exposición alta porque concentran vivienda permanente o población vulnerable. Comercial, Servicios e Industrial son de exposición intermedia.

| Nivel de prioridad | Exposición alta | Exposición intermedia | Uso Otro | Sin uso definido |
|---|---|---|---|---|
| 1 Muy bajo | Sin intervención | Sin intervención | Sin intervención | Condicionada |
| 2 Bajo | Complementaria | Complementaria | Condicionada | Condicionada |
| 3 Moderado | Recomendable | Recomendable | Condicionada | Condicionada |
| 4 Alto | Prioritaria | Recomendable | Condicionada | Condicionada |
| 5 Muy alto | Prioritaria | Prioritaria | Condicionada | Condicionada |

In [ ]:
areas = "areas_analisis"
arcpy.management.CopyFeatures(fc_areas_actividad, areas)
arcpy.analysis.PairwiseErase(fc_expansion, fc_areas_actividad, "expansion_sin_uso")
arcpy.management.Append("expansion_sin_uso", areas, "NO_TEST")

arcpy.conversion.RasterToPolygon("prioridad_nivel", "nivel_poligonos", "NO_SIMPLIFY", "VALUE")
arcpy.analysis.PairwiseIntersect(["nivel_poligonos", areas], "nivel_uso")

In [ ]:
definicion_uso = [f for f in arcpy.ListFields(fc_areas_actividad) if f.name == campo_uso][0]
dominio_uso = {}
if definicion_uso.domain:
    dominio_uso = [d for d in arcpy.da.ListDomains(gdb) if d.name == definicion_uso.domain][0].codedValues
print(dominio_uso)

exposicion_alta = {"residencial", "mixto", "institucional", "dotacional", "institucional dotacional"}


def categoria(nivel, uso):
    uso = dominio_uso.get(uso, uso)
    if uso is None or not str(uso).strip():
        return "Intervención condicionada"
    uso = str(uso).strip().lower().replace("_", " ")
    if uso == "otro":
        return "Sin intervención" if nivel == 1 else "Intervención condicionada"
    if nivel == 1:
        return "Sin intervención"
    if nivel == 2:
        return "Intervención complementaria"
    if nivel == 3:
        return "Intervención recomendable"
    if nivel == 4:
        return "Intervención prioritaria" if uso in exposicion_alta else "Intervención recomendable"
    return "Intervención prioritaria"


arcpy.management.AddField("nivel_uso", "categoria", "TEXT", field_length=40)
arcpy.management.AddField("nivel_uso", "area_ha", "DOUBLE")

with arcpy.da.UpdateCursor("nivel_uso", ["gridcode", campo_uso, "categoria"]) as cursor:
    for nivel_pixel, uso, _ in cursor:
        cursor.updateRow([nivel_pixel, uso, categoria(nivel_pixel, uso)])

arcpy.management.CalculateGeometryAttributes("nivel_uso", [["area_ha", "AREA"]], area_unit="HECTARES")
arcpy.analysis.Statistics("nivel_uso", "resumen_categoria", [["area_ha", "SUM"]], "categoria")

with arcpy.da.SearchCursor("resumen_categoria", ["categoria", "SUM_area_ha"]) as cursor:
    for fila in sorted(cursor, key=lambda f: -f[1]):
        print("{:<32}{:>12,.2f} ha".format(*fila))

In [ ]:
dejar_solo(["prioridad_continua"])
agregar("nivel_uso")

colores = {
    "Sin intervención": [56, 168, 0],
    "Intervención complementaria": [214, 232, 120],
    "Intervención recomendable": [255, 211, 127],
    "Intervención prioritaria": [230, 0, 0],
    "Intervención condicionada": [178, 178, 178]}

capa = mapa.listLayers("nivel_uso")[0]
simbologia = capa.symbology
simbologia.updateRenderer("UniqueValueRenderer")
simbologia.renderer.fields = ["categoria"]
for item in simbologia.renderer.groups[0].items:
    if item.label in colores:
        item.symbol.color = {"RGB": colores[item.label] + [100]}
        item.symbol.outlineColor = {"RGB": [0, 0, 0, 0]}
        item.symbol.outlineWidth = 0
capa.symbology = simbologia

## 4. Concentración espacial de la prioridad

El índice continuo se recorta a las áreas de análisis, se agrega a 90 m con el promedio (cercano a la resolución nativa de la banda térmica) y se convierte a puntos. Sobre esos puntos se aplica Optimized Hot Spot Analysis, que calcula el estadístico Getis-Ord Gi* con una distancia de análisis definida automáticamente y corrección por tasa de falsos descubrimientos (FDR).

$$G_i^* = \frac{\sum_{j} w_{ij}x_j - \bar{X}\sum_{j} w_{ij}}{S\sqrt{\dfrac{n\sum_{j} w_{ij}^2 - \left(\sum_{j} w_{ij}\right)^2}{n-1}}}$$

El campo `Gi_Bin` toma valores de -3 a 3. El valor 3 corresponde a puntos calientes con 99 % de confianza. Esos puntos se convierten a polígonos y recortan las áreas de intervención prioritaria.

In [ ]:
prioridad_areas = ExtractByMask("prioridad_continua", areas)
Aggregate(prioridad_areas, 3, "MEAN").save("prioridad_90m")

arcpy.env.snapRaster = "prioridad_90m"
arcpy.conversion.RasterToPoint("prioridad_90m", "prioridad_puntos", "VALUE")
arcpy.stats.OptimizedHotSpotAnalysis("prioridad_puntos", "puntos_hotspot", "grid_code")

arcpy.analysis.Select("puntos_hotspot", "puntos_calientes_99", "Gi_Bin = 3")
dejar_solo(["nivel_uso"])
capa_hot = agregar("puntos_hotspot")

colores_hot = {
    "-3": [33, 102, 172],
    "-2": [103, 169, 207],
    "-1": [209, 229, 240],
    "0": [200, 200, 200],
    "1": [253, 219, 199],
    "2": [244, 165, 130],
    "3": [215, 25, 28]}

simbologia = capa_hot.symbology
simbologia.updateRenderer("UniqueValueRenderer")
simbologia.renderer.fields = ["Gi_Bin"]
for item in simbologia.renderer.groups[0].items:
    clave = str(item.values[0][0])
    if clave in colores_hot:
        item.symbol.color = {"RGB": colores_hot[clave] + [100]}
        item.symbol.outlineColor = {"RGB": [0, 0, 0, 0]}
        item.symbol.size = 4
capa_hot.symbology = simbologia
arcpy.conversion.PointToRaster("puntos_calientes_99", "Gi_Bin", "hotspot_raster", "MAXIMUM", "NONE", 90)
arcpy.conversion.RasterToPolygon("hotspot_raster", "hotspot_poligonos", "NO_SIMPLIFY", "VALUE")

arcpy.analysis.Select("nivel_uso", "intervencion_prioritaria", "categoria = 'Intervención prioritaria'")
arcpy.analysis.PairwiseClip("intervencion_prioritaria", "hotspot_poligonos", "prioritaria_hotspot")

## 5. Unidad mínima de mapeo y núcleos de intervención

La unidad mínima de mapeo es de 1 ha, equivalente a un píxel de 100 m de la banda térmica. Las áreas prioritarias en puntos calientes se unen sin considerar el uso del suelo, se agrupan los polígonos separados por menos de 20 m (el ancho de una vía local) y se descartan los menores a 10.000 m². Como la agregación rellena los espacios entre polígonos, los núcleos se intersectan de nuevo con las áreas prioritarias para que la superficie reportada sea solo la prioritaria.

Cada área recibe un nivel de atención según su prioridad térmica: Nivel 1 para prioridad muy alta y Nivel 2 para prioridad alta.

In [ ]:
arcpy.analysis.PairwiseDissolve("prioritaria_hotspot", "prioritaria_union", multi_part="SINGLE_PART")
arcpy.cartography.AggregatePolygons(
    "prioritaria_union", "nucleos_agregados",
    aggregation_distance="20 Meters", minimum_area="10000 SquareMeters")

arcpy.management.AddField("nucleos_agregados", "nucleo_id", "LONG")
arcpy.management.CalculateField("nucleos_agregados", "nucleo_id", "!OBJECTID!", "PYTHON3")
arcpy.analysis.PairwiseIntersect(["nucleos_agregados", "prioritaria_hotspot"], "nucleos_prioritarios")

arcpy.management.AddField("nucleos_prioritarios", "nivel_atencion", "SHORT")
arcpy.management.CalculateField("nucleos_prioritarios", "nivel_atencion", "1 if !gridcode! == 5 else 2", "PYTHON3")
arcpy.analysis.PairwiseDissolve("nucleos_prioritarios", "nucleos_intervencion", ["nucleo_id", "nivel_atencion"])
dejar_solo(["nivel_uso", "puntos_hotspot"])
capa_nuc = agregar("nucleos_intervencion")

colores_nivel = {
    "1": [168, 0, 0],
    "2": [255, 140, 0]}
etiquetas_nivel = {
    "1": "Nivel 1: prioridad muy alta",
    "2": "Nivel 2: prioridad alta"}

simbologia = capa_nuc.symbology
simbologia.updateRenderer("UniqueValueRenderer")
simbologia.renderer.fields = ["nivel_atencion"]
for item in simbologia.renderer.groups[0].items:
    clave = str(item.values[0][0])
    if clave in colores_nivel:
        item.symbol.color = {"RGB": colores_nivel[clave] + [100]}
        item.symbol.outlineColor = {"RGB": [0, 0, 0, 0]}
        item.symbol.outlineWidth = 0
        item.label = etiquetas_nivel[clave]
capa_nuc.symbology = simbologia

In [ ]:
def hectareas(fc):
    with arcpy.da.SearchCursor(fc, ["SHAPE@AREA"]) as cursor:
        return sum(fila[0] for fila in cursor) / 10000


arcpy.management.MakeFeatureLayer("nucleos_intervencion", "nivel_1", "nivel_atencion = 1")
arcpy.management.MakeFeatureLayer("nucleos_intervencion", "nivel_2", "nivel_atencion = 2")

etapas = [
    ("Áreas de actividad y expansión", areas),
    ("Intervención prioritaria", "intervencion_prioritaria"),
    ("En puntos calientes al 99 %", "prioritaria_hotspot"),
    ("Núcleos de al menos 1 ha", "nucleos_prioritarios"),
    ("Nivel 1", "nivel_1"),
    ("Nivel 2", "nivel_2")]

for nombre, fc in etapas:
    print("{:<34}{:>12,.2f} ha".format(nombre, hectareas(fc)))

## 6. Comparación de escenarios

Para estudiar cómo cambia la temperatura de superficie entre épocas se calcula la LST de escenas de referencia y se resta de la escena de estudio. La comparación solo es válida si se cumplen estas condiciones:

- Misma época del año, mismo sensor y nubosidad baja.
- Mismo producto y mismo procesamiento en todas las fechas, de modo que la LST se calcule igual.
- Se compara la LST continua, no las clases por cortes naturales, porque los cortes cambian en cada imagen.

$$\Delta LST = LST_{estudio} - LST_{referencia}$$

Un valor positivo indica que la superficie de la escena de estudio estuvo más caliente que la de referencia. La diferencia se resume por núcleo de intervención con estadística zonal. Con pocas referencias el resultado es indicativo y no una climatología.

In [ ]:
ruta_escenas = r"D:\RUTA\DEMO_Fenomeno_ElNino\LC09_L2SP"
referencias = {
    "2025": os.path.join(ruta_escenas, "escena2025"),
    "2024": os.path.join(ruta_escenas, "escena2024")}

arcpy.env.snapRaster = "lst_base"
arcpy.analysis.PairwiseDissolve("nucleos_intervencion", "nucleos_zonas", ["nucleo_id"])

for anio, carpeta in referencias.items():
    quitar_capas(["lst_dif_" + anio])
    ndvi_ref, lst_ref = calcular_escena(carpeta, "ref" + anio)
    (Raster("lst_base") - Raster(lst_ref)).save("lst_dif_" + anio)
    arcpy.sa.ZonalStatisticsAsTable("nucleos_zonas", "nucleo_id", "lst_dif_" + anio,
                                    "dif_nucleo_" + anio, "DATA", "MEAN")

dejar_solo(["nucleos_intervencion"])
mostrar("lst_dif_2025")

for anio in referencias:
    print("Diferencia de LST por núcleo frente a la referencia de " + anio)
    with arcpy.da.SearchCursor("dif_nucleo_" + anio, ["nucleo_id", "MEAN"]) as cursor:
        for nucleo, media in sorted(cursor, key=lambda f: -f[1])[:10]:
            print("Núcleo {:>4}: {:+.2f} °C".format(nucleo, media))

## Referencias

- Robayo Puerto, M. F. y Ruiz Buitrago, M. Y. (2026). Modelo espacial multicriterio para la identificación de zonas idóneas para la incorporación de infraestructura verde como mitigadora de islas de calor urbanas en el marco del POT de Barranquilla. Universidad Antonio Nariño.
- USGS. Landsat 8-9 Collection 2 Level 2 Science Product Guide (LSDS-1619).
- Saaty, T. L. (1977). A scaling method for priorities in hierarchical structures. Journal of Mathematical Psychology, 15(3), 234-281.
- Getis, A. y Ord, J. K. (1992). The analysis of spatial association by use of distance statistics. Geographical Analysis, 24(3), 189-206.
- Ord, J. K. y Getis, A. (1995). Local spatial autocorrelation statistics: distributional issues and an application. Geographical Analysis, 27(4), 286-306.